# Traffic Congestion Prediction & Smart Traffic Analytics

This notebook contains the exploratory data analysis (EDA), feature engineering, model training, and evaluation for the Traffic Congestion Prediction & Smart Traffic Analytics System.

## 1. Setup and Environment

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set path relative to project root
os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

## 2. Load and Inspect Dataset

In [ ]:
from src.data_loader import load_raw_data
df_raw = load_raw_data()
df_raw.head()

In [ ]:
df_raw.info()
print(f"Missing values:\n{df_raw.isnull().sum()}")

## 3. Data Preprocessing

In [ ]:
from src.preprocessing import preprocess_data
df_clean = preprocess_data(df_raw)
df_clean.head()

## 4. Feature Engineering

In [ ]:
# Split chronologically (80% train, 20% test) before fitting profiles to avoid leakage
split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx].copy()
test_df = df_clean.iloc[split_idx:].copy()

from src.feature_engineering import fit_historical_profiles, build_features
profiles = fit_historical_profiles(train_df)
train_df_feat = build_features(train_df, profiles)
test_df_feat = build_features(test_df, profiles)
train_df_feat.head()

## 5. Exploratory Data Analysis

In [ ]:
# Hourly Traffic Volume by Junction
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_clean, x="Hour", y="Vehicles", hue="Junction", palette="tab10")
plt.title("Average Hourly Traffic Volume by Junction")
plt.xlabel("Hour of Day")
plt.ylabel("Average Vehicles")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Weekday vs Weekend Traffic
plt.figure(figsize=(8, 5))
sns.barplot(data=df_clean, x="Junction", y="Vehicles", hue="IsWeekend", palette="Set2")
plt.title("Traffic Comparison: Weekday (0) vs Weekend (1)")
plt.xlabel("Junction ID")
plt.ylabel("Average Vehicles")
plt.grid(True, alpha=0.3, axis="y")
plt.show()

## 6. Model Training and Comparison

In [ ]:
# Import custom training functions
from src.train import make_model_ready_data
X_train, y_train, feature_cols, scaler = make_model_ready_data(train_df_feat, is_train=True)

preprocessor = {
    "scaler": scaler,
    "feature_names": feature_cols,
    "historical_profiles": profiles
}
X_test, y_test, _, _ = make_model_ready_data(test_df_feat, is_train=False, preprocessor=preprocessor)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from src.evaluate import calculate_metrics

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    m = calculate_metrics(y_test, y_pred)
    print(f"{name}: MAE={m['MAE']:.3f}, RMSE={m['RMSE']:.3f}, R2={m['R2']:.3f}")

## 7. Run Predictions

In [ ]:
from src.predict import TrafficPredictor
predictor = TrafficPredictor()
# Note: Call predictor.load_model() after training via the CLI
# predictor.load_model()
# predictor.predict(pd.Timestamp('2026-10-12 08:00:00'), 1)